![](https://github.com/ibmm-unibe-ch/FrankenMSA/blob/dev/app/assets/frankenmsa_header.png?raw=true)

# FrankenMSA-Local
This notebook launches the [FrankenMSA App](https://github.com/ibmm-unibe-ch/FrankenMSA/tree/main/) **in a Local Environment** to provide a GUI for manipulating Multiple Sequence Alignments (MSAs).


Tip: use “Runtime” → “Run all” (or `Ctrl + F9`) to execute all cells.

In [1]:
#@title Install Prerequisites

INSTALL_PREREQUISITES = False  #@param {type:"boolean"}

import os

if INSTALL_PREREQUISITES:
    # Plain Dash only; removed jupyter_dash
    os.system("pip install termcolor gitpython ipywidgets ipython dotenv > /dev/null 2>&1")

import git, sys, importlib
from pathlib import Path
from termcolor import colored
import dotenv

dotenv.load_dotenv()
os.environ["ON_COLAB"] = "0"

def warn(msg):
    print(colored("[WARNING] ", "yellow") + msg, file=sys.stderr)

def info(msg):
    print(colored("[INFO] ", "cyan") + msg)

if INSTALL_PREREQUISITES:
    info("Installed prerequisite packages.")

In [ ]:
#@title Prepare Rendering and Sharing options

import ipywidgets as widgets
from IPython.display import display

_render_dropdown = widgets.Dropdown(
    options=[
        ("External browser tab", "external"),
        ("Inline (within notebook)", "inline"),
    ],
    value="external",
    description="Render:",
)
_ngrok_checkbox = widgets.Checkbox(
    value=False,
    description="Create ngrok share link",
)
_port_input = widgets.IntText(
    value=8050,
    description="Port:",
    min=1024,
    max=65535,
)
_status = widgets.Output()

use_ngrok = lambda : _ngrok_checkbox.value

def _enforce_render_on_ngrok(change):
    if change["name"] != "value":
        return
    with _status:
        _status.clear_output()
        if change["new"]:
            if _render_dropdown.value != "external":
                _render_dropdown.value = "external"
            print("ngrok sharing forces external rendering.")
        else:
            print("ngrok sharing disabled; inline available.")

def _enforce_ngrok_on_render(change):
    if change["name"] != "value":
        return
    if use_ngrok() and change["new"] != "external":
        with _status:
            _status.clear_output()
            print("ngrok sharing forces external rendering.")
        _render_dropdown.value = "external"

_ngrok_checkbox.observe(_enforce_render_on_ngrok, names="value")
_render_dropdown.observe(_enforce_ngrok_on_render, names="value")
display(widgets.VBox([_render_dropdown, _ngrok_checkbox, _port_input, _status]))

In [3]:
#@title Install FrankenMSA

INSTALL_FRANKENMSA = False #@param {type:"boolean"}

if INSTALL_FRANKENMSA:
    setup_py = Path("setup.py")
    if setup_py.exists():
        os.system("pip install -e . > /dev/null 2>&1")
        info("FrankenMSA installed.")
    else:
        warn("FrankenMSA setup.py not found; this does not look like the FrankenMSA repository. Skipping installation.")
        FRANKEN_GIT_URL = "https://github.com/ibmm-unibe-ch/FrankenMSA.git"
        FRANKEN_GIT_BRANCH = input("Enter the FrankenMSA branch to clone (leave empty for default: main): ").strip() or "main"
        os.system(f"git clone --branch {FRANKEN_GIT_BRANCH} {FRANKEN_GIT_URL} FrankenMSA > /dev/null 2>&1")
        os.system("cd FrankenMSA; pip install -e . > /dev/null 2>&1; cd -")
    info("FrankenMSA installation complete.")
    
# kill any existing instances
!pkill -f "app/app.py" 2>/dev/null || true
!pkill -f "gunicorn" 2>/dev/null || true
!pkill -f "ngrok" 2>/dev/null || true

In [8]:
#@title Launch FrankenMSA App (without ngrok sharing)
if not use_ngrok():
    use_inline = _render_dropdown.value == "inline"
    if use_inline:
        from app.app import launch
        launch(
            render_mode=_render_dropdown.value,
            port=_port_input.value,
        )
    else:

        import subprocess, time

        # setup environment variables for subprocess app launch
        PORT = _port_input.value
        env = os.environ.copy()
        env["PORT"], env["HOST"] = str(PORT), os.environ.get("HOST", "127.0.0.1")
        env["PYTHONPATH"] = env.get("PYTHONPATH", "")
        env["FRANKEN_COLAB"] = "0"
        env["ON_COLAB"] = "0"
        env["FRANKEN_RENDER_MODE"] = _render_dropdown.value
        env.pop("COLAB_TUNNEL_URL", None)

        proc = subprocess.Popen(
            [sys.executable, "app/app.py"],
            cwd=str(Path.cwd().resolve()),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        start = time.time()
        lines = []
        while time.time() - start < 25:
            ln = proc.stdout.readline()
            if ln:
                lines.append(ln.rstrip())
                if "Running on" in ln or "Dash is running" in ln:
                    break
            else:
                time.sleep(0.2)

        display_host = env["HOST"] if env["HOST"] not in {"0.0.0.0", "::"} else "0.0.0.0"
        open_target = f"http://{display_host}:{PORT}"

        print("\n---- recent logs ----")
        print("\n".join(lines[-20:]))
        print("---------------------")
        info(f"🌐 Open: {open_target}")
        # print("✅ Look for the banner 'IF-build:xxxx' on the page header to confirm version")
        print("📡 Tailing FrankenMSA app logs (Ctrl+C to stop):")
        while True:
            line = proc.stdout.readline()
            if not line:
                time.sleep(0.2)
                continue
            print(line, end="")


---- recent logs ----
[UPLOAD_DIR] using: /Users/noahhk/.frankenmsa/uploads
Dash starting on http://127.0.0.1:8050
Dash is running on http://127.0.0.1:8050/
---------------------
[INFO] 🌐 Open: http://127.0.0.1:8050
📡 Tailing FrankenMSA app logs (Ctrl+C to stop):

 * Serving Flask app 'app'
 * Debug mode: off
 * Running on http://127.0.0.1:8050
Press CTRL+C to quit
127.0.0.1 - - [21/Nov/2025 15:43:08] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [21/Nov/2025 15:43:08] "GET /assets/styles.css?m=1763725155.452639 HTTP/1.1" 304 -
127.0.0.1 - - [21/Nov/2025 15:43:08] "GET /assets/animated_background.js?m=1747911889.3687897 HTTP/1.1" 304 -
127.0.0.1 - - [21/Nov/2025 15:43:08] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [21/Nov/2025 15:43:08] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [21/Nov/2025 15:43:08] "GET /assets/icon_main_white_transparent.png HTTP/1.1" 304 -
127.0.0.1 - - [21/Nov/2025 15:43:08] "GET /assets/icon_files_white_transparent.png HTTP/1.1" 304 -
127.0.0.1 - - [21

KeyboardInterrupt: 

In [9]:
#@title Launch FrankenMSA App (with ngrok sharing)
if use_ngrok():

    try:
        import pyngrok
    except:
        info("Installing pyngrok...")
        os.system("pip install pyngrok > /dev/null 2>&1")
        info("pyngrok installed.")
    try:
        from pyngrok import ngrok

    except:
        raise ImportError("Pyngrok could not be installed")

    from pyngrok import ngrok, conf
    import getpass, re
    import dotenv
    dotenv.load_dotenv()
    
    token = os.environ.get("NGROK_AUTH_TOKEN", "").strip()
    if not token:
        warn("No ngrok auth token found in NGROK_AUTH_TOKEN env variable.")
        warn("You can sign up for a free ngrok account at https://ngrok.com/")
        warn("To avoid entering the token every time, the token is set it in the NGROK_AUTH_TOKEN environment variable after entering.")

        token = getpass.getpass("Enter ngrok authtoken (hidden): ").strip().strip("'").strip('"')
        os.environ["NGROK_AUTH_TOKEN"] = token

        info("ngrok auth token set as environment variable.")
    conf.get_default().auth_token = token

    public_url = None
    PORT = _port_input.value
    try:
        for t in ngrok.get_tunnels():
            addr = (t.config or {}).get("addr", "")
            if addr.endswith(f":{PORT}"):
                public_url = t.public_url
                print("♻️ Reusing existing tunnel:", public_url)
                break

        if not public_url:
            tun = ngrok.connect(addr=f"0.0.0.0:{PORT}", proto="http")
            public_url = tun.public_url
            print("✅ Created new tunnel:", public_url)

    except Exception as e:
        msg = str(e)
        m = re.search(r"https?://[a-z0-9\-]+\.ngrok-[\w\-]+\.(?:dev|app)", msg)
        if m:
            public_url = m.group(0)
            warn("♻️ Using tunnel from error message:", public_url)
        else:
            raise e

    import subprocess, time

    # setup environment variables for subprocess app launch
    env = os.environ.copy()
    env["PORT"], env["HOST"] = str(PORT), os.environ.get("HOST", "0.0.0.0")
    env["PYTHONPATH"] =  env.get("PYTHONPATH", "")
    env["FRANKEN_COLAB"] = "0"
    env["ON_COLAB"] = "0"
    env["FRANKEN_RENDER_MODE"] = _render_dropdown.value
    env["FRANKEN_USE_NGROK"] = "1" if _ngrok_checkbox.value else "0"
    if public_url:
        env["COLAB_TUNNEL_URL"] = public_url
    else:
        env.pop("COLAB_TUNNEL_URL", None)

    proc = subprocess.Popen(
        [sys.executable, "app/app.py"],
        cwd=str(Path.cwd().resolve()),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    start = time.time()
    lines = []
    while time.time() - start < 25:
        ln = proc.stdout.readline()
        if ln:
            lines.append(ln.rstrip())
            if "Running on" in ln or "Dash is running" in ln:
                break
        else:
            time.sleep(0.2)

    display_host = env["HOST"] if env["HOST"] not in {"0.0.0.0", "::"} else "0.0.0.0"
    open_target = public_url if public_url else f"http://{display_host}:{PORT}"

    print("\n---- recent logs ----")
    print("\n".join(lines[-20:]))
    print("---------------------")
    info(f"🌐 Open: {open_target}")
    # print("✅ Look for the banner 'IF-build:xxxx' on the page header to confirm version")
    print("📡 Tailing FrankenMSA app logs (Ctrl+C to stop):")
    while True:
        line = proc.stdout.readline()
        if not line:
            time.sleep(0.2)
            continue
        print(line, end="")

[WARNING] No ngrok auth token found in NGROK_AUTH_TOKEN env variable.
[WARNING] You can sign up for a free ngrok account at https://ngrok.com/
[WARNING] To avoid entering the token every time, the token is set it in the NGROK_AUTH_TOKEN environment variable after entering.


[INFO] ngrok auth token set as environment variable.
✅ Created new tunnel: https://gracelynn-uncatastrophic-organizingly.ngrok-free.dev

---- recent logs ----
[UPLOAD_DIR] using: /Users/noahhk/.frankenmsa/uploads
Dash starting on http://0.0.0.0:8056
🌐 Public tunnel: https://gracelynn-uncatastrophic-organizingly.ngrok-free.dev
Dash is running on http://0.0.0.0:8056/
---------------------
[INFO] 🌐 Open: https://gracelynn-uncatastrophic-organizingly.ngrok-free.dev
📡 Tailing FrankenMSA app logs (Ctrl+C to stop):

 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8056
 * Running on http://10.3.1.236:8056
Press CTRL+C to quit
127.0.0.1 - - [21/Nov/2025 15:44:26] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [21/Nov/2025 15:44:26] "GET /assets/styles.css?m=1763725155.452639 HTTP/1.1" 304 -
127.0.0.1 - - [21/Nov/2025 15:44:26] "GET /assets/animated_background.js?m=1747911889.3687897 HTTP/1.1" 304 -
127.0.0.1 - - [21/Nov/2025 15:44:27] "GE

KeyboardInterrupt: 